# Wildfire ignition **dataset** — walkthrough

This notebook walks through the three pipeline stages that build the labelled
fire / no-fire dataset, driven by the `firepredict` package: **clean fires →
add weather → add terrain**. It reads the CSVs each stage writes under
`outputs/processed/` and shows how to rebuild them. Training a model on the
final CSV is out of scope for this repo.

## 0 · Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Put the project root on sys.path so `import firepredict` works whether
# Jupyter was launched from the project root or from notebooks/.
_here = Path.cwd().resolve()
PROJECT_ROOT = _here.parent if _here.name == "notebooks" else _here
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from firepredict import config

print("PROJECT_ROOT  :", config.PROJECT_ROOT)
print("DATA_DIR      :", config.DATA_DIR)
print("PROCESSED_DIR :", config.PROCESSED_DIR)
print("FIGURES_DIR   :", config.FIGURES_DIR)
print("CACHE_DIR     :", config.CACHE_DIR)

## 1 · Stage 1 — clean fires (`stage1_clean_fires`)

`firepredict.pipeline.stage1_clean_fires` merges the SGIF Excel registries with the ICNF burned-area shapefiles, dedupes by `Cod_SGIF`, reprojects geometries to EPSG:4326, derives `lat`/`lon` from each polygon centroid, and reconstructs missing `DH_Inicio` timestamps from the `Ano/Mes/Dia/Hora` SGIF columns. Output: `outputs/processed/cleaned_fires.csv`.

In [ ]:
fires = pd.read_csv(config.CLEANED_FIRES_CSV, low_memory=False)
fires["DH_Inicio"] = pd.to_datetime(fires["DH_Inicio"], errors="coerce")

print(f"Total fires : {len(fires):,}")
print(f"Date range  : {fires['DH_Inicio'].min()} → {fires['DH_Inicio'].max()}")
print()
print("Causa_Tipo distribution:")
print(fires["Causa_Tipo"].value_counts(dropna=False))
fires.head()

**Rebuild from raw `data/`** (a few minutes — re-reads SGIF Excels and shapefiles):

In [ ]:
# from firepredict.pipeline import stage1_clean_fires
# stage1_clean_fires.main()

## 2 · Stage 2 — Open-Meteo weather (`stage2_add_weather`)

`firepredict.pipeline.stage2_add_weather` calls the Open-Meteo Archive API for each fire and writes two CSVs:

- `fire_weather_dataset_2024.csv` — temp / humidity / wind / precip at the fire's start hour
- `fire_weather_dataset_timeseries_2024.csv` — the 24 h preceding each fire flattened into `temp_t-24..t-1`, `hum_t-24..t-1`, `wind_t-24..t-1`

The HTTP client is wrapped in `requests_cache` (infinite TTL), so any rerun is served from `.cache/openmeteo.sqlite` rather than re-hitting the API.

In [ ]:
weather = pd.read_csv(config.WEATHER_POINT_CSV, low_memory=False)
print(f"Rows: {len(weather):,}")
print()
print("Non-null counts for the new weather columns:")
print(weather[["temp", "humidity", "wind_speed", "precip"]].notna().sum())
weather[["Cod_SGIF", "DH_Inicio", "lat", "lon", "temp", "humidity", "wind_speed", "precip"]].head()

In [ ]:
# Demonstrate the per-row weather fetcher and confirm the cache is alive.
from firepredict.weather import build_client, get_weather_at_fire

client = build_client()
sample = weather.iloc[0]
print(f"Sample fire: {sample['Cod_SGIF']} at {sample['DH_Inicio']}  ({sample['lat']:.3f}, {sample['lon']:.3f})")
print()
print("get_weather_at_fire result (cache-served):")
print(get_weather_at_fire(client, sample))

In [ ]:
# Quick peek at the 24h-lookback timeseries CSV.
ts = pd.read_csv(config.WEATHER_TIMESERIES_CSV, low_memory=False)
ts_cols = [c for c in ts.columns if c.startswith(("temp_t-", "hum_t-", "wind_t-"))]
print(f"Rows: {len(ts):,}")
print(f"Lookback feature columns: {len(ts_cols)} (e.g. {ts_cols[:3]} ... {ts_cols[-3:]})")
ts[["Cod_SGIF", "DH_Inicio"] + ts_cols[:6]].head()

**Rebuild both weather stages** (slow on first run, instant on rerun thanks to the cache):

In [ ]:
# from firepredict.pipeline import stage2_add_weather
# stage2_add_weather.main()

## 3 · Stage 3 — terrain features (`stage3_add_terrain`)

`firepredict.pipeline.stage3_add_terrain` samples three Copernicus rasters (`slope`, `roughness`, `aspect`) at every fire point and writes `final_fire_weather_terrain_2024.csv`. Aspect (0-360°) is also encoded as `aspect_sin` / `aspect_cos` so a model can treat it as a circular variable (358° and 2° really are close).

In [ ]:
final = pd.read_csv(config.TERRAIN_FINAL_CSV, low_memory=False)
terrain_cols = ["slope", "roughness", "aspect", "aspect_sin", "aspect_cos"]

print(f"Rows: {len(final):,}")
print()
print("Terrain column summary:")
print(final[terrain_cols].describe())
final[["Cod_SGIF", "lat", "lon"] + terrain_cols].head()

**Rebuild** (seconds — pure raster sampling, no API calls):

In [ ]:
# from firepredict.pipeline import stage3_add_terrain
# stage3_add_terrain.main()

## Where artifacts went

- CSVs:  `outputs/processed/` — the final dataset is `final_fire_weather_terrain_*.csv`
- Cache: `.cache/openmeteo.sqlite` (only when the optional Open-Meteo backend is used)

To run the whole dataset build non-interactively from a shell:

```bash
.venv/bin/python -m firepredict.pipeline
```

Or individual stages:

```bash
.venv/bin/python -m firepredict.pipeline.stage1_clean_fires
.venv/bin/python -m firepredict.pipeline.stage2_add_weather
.venv/bin/python -m firepredict.pipeline.stage3_add_terrain
```